# AI-Powered Intrusion Detection System (CLO 4)

This notebook implements a proof-of-concept IDS classifier for **SecureNet Corp** using **Gradient Boosting** on CIC-IDS2017 style flow data.

## Workflow
1. Load raw IDS CSV files from `data/raw`
2. Preprocess data (missing/infinite handling, encoding, scaling, split)
3. Train Gradient Boosting classifier
4. Evaluate using security metrics (accuracy, precision, recall, confusion matrix)
5. Interpret security implications

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ids_pipeline import evaluate_model, load_raw_dataset, preprocess_dataset, train_model

sns.set_theme(style="whitegrid")

In [ ]:
DATA_DIR = ROOT / "data" / "raw"
OUTPUT_PATH = ROOT / "data" / "processed" / "notebook_metrics.json"

print(f"Working directory: {ROOT}")
print(f"Data directory: {DATA_DIR}")
print("Tip: Place official CIC-IDS2017 CSV files inside data/raw for full-scale execution.")

In [ ]:
df = load_raw_dataset(DATA_DIR)
print(f"Loaded rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()

In [ ]:
label_col = "Label" if "Label" in df.columns else [c for c in df.columns if c.lower() == "label"][0]
class_counts = df[label_col].astype(str).value_counts().head(10)
display(class_counts.to_frame(name="count"))

plt.figure(figsize=(10, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Top Label Distribution")
plt.xlabel("Traffic Label")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Preprocessing
The pipeline applies:
- Infinite value replacement and numeric imputation
- Categorical feature encoding
- Standard scaling
- Stratified train/test split

In [ ]:
bundle = preprocess_dataset(df)
print("X_train shape:", bundle.X_train.shape)
print("X_test shape :", bundle.X_test.shape)
print("Positive class in train:", bundle.y_train.mean().round(4))
print("Positive class in test :", bundle.y_test.mean().round(4))

## Model Design Choice
Gradient Boosting is selected because it captures non-linear feature interactions in network flow data while maintaining strong practical accuracy for supervised IDS classification.

In [ ]:
model = train_model(bundle)
metrics = evaluate_model(model, bundle)

print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Attack Precision: {metrics['precision_attack']:.4f}")
print(f"Attack Recall: {metrics['recall_attack']:.4f}")

In [ ]:
report_df = pd.DataFrame(metrics['classification_report']).transpose()
display(report_df)

cm = metrics['confusion_matrix']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Attack"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=bundle.feature_names).sort_values(ascending=False).head(12)
display(importance.to_frame(name="importance"))

plt.figure(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index, palette="mako")
plt.title("Top 12 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## Security Interpretation
- High attack recall means fewer malicious flows are missed.
- Any drop in attack recall increases risk of undetected intrusion.
- False positives still require analyst time, so threshold tuning is important in production.
- Recommended enhancement: periodic retraining on fresh traffic and complementary anomaly detection for zero-day behavior.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    'rows_used': int(len(df)),
    'num_features': int(len(bundle.feature_names)),
    'metrics': metrics
}
OUTPUT_PATH.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(f"Saved notebook metrics to: {OUTPUT_PATH}")